# 02 – Feature Encoding & Selection

**Owner: M3 – Primesh Marasingha**

This notebook demonstrates the encoding pipeline defined in `src/encoders.py`,
verifies correct fit-only-on-train behaviour, and calls `select_features()` to
identify the most predictive columns.

**Prerequisites**: run `make split` first (requires M1's `data_prep.py`).


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_PROCESSED, CATEGORICAL_LOW, CATEGORICAL_HIGH, NUMERIC, TARGET
)
from src.encoders import build_encoder, select_features

sns.set_theme(style='whitegrid')
%matplotlib inline
print('imports OK')

## 1. Load split


In [ ]:
X_train = pd.read_parquet(DATA_PROCESSED / 'X_train.parquet')
X_test  = pd.read_parquet(DATA_PROCESSED / 'X_test.parquet')
y_train = pd.read_parquet(DATA_PROCESSED / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(DATA_PROCESSED / 'y_test.parquet').squeeze()

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Positive rate  train: {y_train.mean():.1%}  test: {y_test.mean():.1%}')

## 2. Column inventory

Quick sanity-check: confirm no leakage columns survived `clean()`.


In [ ]:
from src.config import LEAKAGE_COLS

leaking = [c for c in LEAKAGE_COLS if c in X_train.columns]
assert not leaking, f'Leakage columns found: {leaking}'
print('No leakage columns — clean.')

print('\nCATEGORICAL_LOW  present:', [c for c in CATEGORICAL_LOW  if c in X_train.columns])
print('CATEGORICAL_HIGH present:', [c for c in CATEGORICAL_HIGH if c in X_train.columns])
print('NUMERIC present         :', [c for c in NUMERIC          if c in X_train.columns])

## 3. One-Hot Encoding (low-cardinality)


In [ ]:
# Inspect cardinality of CATEGORICAL_LOW
for col in CATEGORICAL_LOW:
    if col in X_train.columns:
        n = X_train[col].nunique()
        print(f'  {col:25s}: {n:3d} unique values')

## 4. Target Encoding (high-cardinality)


In [ ]:
# Confirm high-cardinality values
for col in CATEGORICAL_HIGH:
    if col in X_train.columns:
        n = X_train[col].nunique()
        print(f'  {col:20s}: {n:5d} unique values')

In [ ]:
# Demonstrate TargetEncoder smoothing on Order City
from sklearn.preprocessing import TargetEncoder

te = TargetEncoder(smooth='auto', target_type='binary')
te.fit(X_train[['Order City']], y_train)

city_sample = X_train[['Order City']].drop_duplicates().head(10)
encoded = te.transform(city_sample)
city_sample = city_sample.copy()
city_sample['P(late|city)'] = encoded[:, 0]
print(city_sample.sort_values('P(late|city)', ascending=False))

## 5. Build & fit the full ColumnTransformer


In [ ]:
encoder = build_encoder(high_card_strategy='target')

# Must fit on train only — test data never touches the fit step
X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc  = encoder.transform(X_test)

feature_names = encoder.get_feature_names_out()
print(f'Encoded shape — train: {X_train_enc.shape}  test: {X_test_enc.shape}')
print(f'Total features: {len(feature_names)}')
print('First 10 feature names:', feature_names[:10])

## 6. Frequency Encoding (alternative)


In [ ]:
freq_encoder = build_encoder(high_card_strategy='frequency')
X_train_freq = freq_encoder.fit_transform(X_train, y_train)
X_test_freq  = freq_encoder.transform(X_test)
print(f'Frequency-encoded shape: {X_train_freq.shape}')

## 7. Feature importance & selection


In [ ]:
# select_features uses a lightweight XGBoost probe + correlation pruning
selected = select_features(
    X_train, y_train,
    fitted_transformer=encoder,
    importance_threshold=0.001,
    corr_threshold=0.95,
)
print(f'\nSelected {len(selected)} features:')
for f in selected:
    print(f'  {f}')

## 8. Feature importance plot


In [ ]:
from xgboost import XGBClassifier

probe = XGBClassifier(
    n_estimators=100, max_depth=4, random_state=42,
    eval_metric='logloss', verbosity=0
)
probe.fit(X_train_enc, y_train)

imp = pd.Series(probe.feature_importances_, index=feature_names)
top20 = imp.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
top20.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Feature importance (gain)')
ax.set_title('Top-20 XGBoost Feature Importances')
plt.tight_layout()
plt.savefig('../reports/feature_importances.png', dpi=150)
plt.show()

## 9. Correlation heatmap (numeric block)


In [ ]:
# Correlation among numeric features only
num_idx = [i for i, n in enumerate(feature_names) if n.startswith('num__')]
num_names = feature_names[num_idx]
X_num = pd.DataFrame(X_train_enc[:, num_idx], columns=num_names)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    X_num.corr(), annot=True, fmt='.2f', cmap='coolwarm',
    center=0, ax=ax, square=True, linewidths=0.5
)
ax.set_title('Correlation – Numeric Features')
plt.tight_layout()
plt.savefig('../reports/numeric_correlation.png', dpi=150)
plt.show()

## 10. Summary

| Encoding block    | Cols in | Cols out | Strategy |
|-------------------|---------|----------|-----------|
| ohe               | 6       | ~60      | OneHotEncoder (handle_unknown=ignore) |
| high_card (target)| 4       | 4        | TargetEncoder (smooth=auto) |
| num               | 11      | 11       | passthrough |
| **Total**         | **21**  | **~75**  | |

The `select_features()` probe further reduces this to the most informative
subset.  The full feature set is used in `src/train_xgb.py`.
